# CASMI 2026 离线提交

添加比赛数据和本地 `export-assets` 导出的 Kaggle Dataset。关闭 Internet。修改下面 ASSETS。此 Notebook 只做推理，不训练，不构建训练缓存。模型可选；纯检索使用 CPU 即可。

In [ ]:
from pathlib import Path
import subprocess, sys

ASSETS = Path('/kaggle/input/your-casmi26-inference-assets')  # 修改为资产目录
DATA_DIR = None  # 若有多个比赛数据目录，填写实际路径
assert (ASSETS / 'infer.py').exists(), f'请检查 ASSETS: {ASSETS}'
assert (ASSETS / 'cache/manifest.json').exists(), '需要完整缓存，请先在本地 prepare + export-assets'

WHEELS = ASSETS / 'wheels'
if WHEELS.exists():
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-index',
        '--find-links', str(WHEELS), '-r', str(ASSETS / 'requirements.txt')])


In [ ]:
command = [sys.executable, str(ASSETS / 'infer.py'), '--assets', str(ASSETS),
           '--output', '/kaggle/working/submission.csv', '--device', 'auto']
if DATA_DIR:
    command += ['--data-dir', str(DATA_DIR)]
subprocess.check_call(command)


In [ ]:
import pandas as pd
submission = pd.read_csv('/kaggle/working/submission.csv')
print('molecules:', len(submission))
print(submission.head().to_string(index=False))
assert submission.columns.tolist() == ['molecule_id', 'smiles']


推理入口已经验证每个当前 test molecule_id 恰好一行、候选不超过 25、SMILES 合法和评分身份不重复。保存并运行禁网版本，检查时长满足比赛限制后 Submit。未提供排行榜成绩；公开测试为格式样例。